In [4]:
import pandas as pd

# Cargar datos
df = pd.read_excel('/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/resultados_distrital.xlsx')

# (Opcional) asegurar que solo tenemos 2018 y 2022
# df = df[df['año'].isin([2018, 2022])]

# ---------------------------------------------------------
# 2. Identificar ganadores 2018 y 2022 por distrito (ubigeo)
# ---------------------------------------------------------
winners = df.loc[
    df.groupby(['ubigeo', 'año'])['total_votos'].idxmax(),
    ['ubigeo', 'año', 'organizacion_politica']
]

# Ganadores 2018
w_2018 = (
    winners[winners['año'] == 2018]
    .rename(columns={'organizacion_politica': 'ganador_2018'})
    [['ubigeo', 'ganador_2018']]
)

# Ganadores 2022
w_2022 = (
    winners[winners['año'] == 2022]
    .rename(columns={'organizacion_politica': 'ganador_2022'})
    [['ubigeo', 'ganador_2022']]
)

# Unir ganadores 2022 con el ganador 2018 del mismo distrito
ganadores_merged = pd.merge(w_2022, w_2018, on='ubigeo', how='left')

# ---------------------------------------------------------
# 3. Crear variable turnover (reelección en 2022)
#    turnover = 1 si la organización política fue reelegida
# ---------------------------------------------------------
ganadores_merged['turnover'] = (
    ganadores_merged['ganador_2022'] == ganadores_merged['ganador_2018']
).astype(int)

# ---------------------------------------------------------
# 4. Trabajar solo con los datos de 2022
# ---------------------------------------------------------
df_2022 = df[df['año'] == 2022].copy()

# 4.1 Probabilidad de ganar (participación en votos en el distrito en 2022)
df_2022['probabilidad_ganar'] = df_2022.groupby('ubigeo')['total_votos'].transform(
    lambda x: x / x.sum()
)

# 4.2 Agregar turnover a la base 2022
df_2022 = df_2022.merge(
    ganadores_merged[['ubigeo', 'turnover']],
    on='ubigeo',
    how='left'
)

# ---------------------------------------------------------
# 5. Guardar base final solo con 2022
# ---------------------------------------------------------

df.to_excel('/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/data_partido/resultados_con_variables.xlsx', index=False)